## Easy record-linkage using the LinkTransformer libary

From https://colab.research.google.com/drive/1OqUB8sqpUvrnC8oa_1RoOUzV6DaAKL4N?usp=sharing#scrollTo=t0BTOG2bqeIQ

## Setup

In [1]:
# An environment with python=3.11 works. (3.12 does not work.)

#!pip install linktransformer

In [2]:
import linktransformer as lt
import pandas as pd
import os


In [3]:
os.chdir('C:/Dropbox/Applied-Economics/task12')

# Supported Pre-trained models

We support several LinkedTransformer models, check out our guide on selecting the best models on this tutorial notebook [model zoo](https://colab.research.google.com/drive/1SAvQdgYiX2CinoTC8Y5qtKScNwx3DYXf?usp=sharing). Our framwork supports all language models on [HuggingFace](https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=trending). We highly recommend the sentence transformers models also available on HuggingFace. For more on sentence-transformers, visit their [website](https://www.sbert.net/)

For users of LinkedTransformer who are new to NLP, here is a guide we have prepared for you to select an appropriate semantic similarity model. (https://colab.research.google.com/drive/1SAvQdgYiX2CinoTC8Y5qtKScNwx3DYXf?usp=sharing)

For the use case of linking companies and harmonising trade products (applications covered in our paper), here is a tutorial on how to do it with a high level interface, check out our package [repo](https://github.com/dell-research-harvard/linktransformer)

## Standard (deep) merge on a key

Just like you would in any merge command!

In [4]:
###Read data
df1 = pd.read_csv("toy_comp_1.csv")
df2 = pd.read_csv("toy_comp_2.csv")

df1.head(3)

,CompanyID,CompanyName,Industry,Founded_Year,Country
0,1,TechCorp,Technology,2005,USA
1,2,InfoTech Solutions,Technology,1998,Canada
2,3,GlobalSoft Inc,Software,2010,India


In [5]:
import linktransformer as lt


# Test your function here
df_lm_matched = lt.merge(df2, df1, merge_type='1:m', on="CompanyName", model="all-MiniLM-L6-v2", left_on=None, right_on=None)

df_lm_matched.head(10)



2024-10-26 08:24:23 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:25 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,CompanyID_x,CompanyName_x,Revenue (Millions USD),Num_Employees,Country_x,id_lt_x,CompanyID_y,CompanyName_y,Industry,Founded_Year,Country_y,id_lt_y,score
0,1,Tech Corporation,5000,10000,USA,0,1,TechCorp,Technology,2005,USA,0,0.828613
1,2,InfoTech Soln,4500,8500,Canada,1,2,InfoTech Solutions,Technology,1998,Canada,1,0.806597
2,3,GlobalSoft Incorporated,3000,6000,India,2,3,GlobalSoft Inc,Software,2010,India,2,0.941341
3,4,DataTech Corporation,2500,5000,Germany,3,4,DataTech Co,Data Analytics,2012,Germany,3,0.934297
4,5,SoftSys Limited,4000,7500,UK,4,5,SoftSys Ltd,Software,2003,UK,4,0.811891
5,6,TechCorp,5500,12000,USA,5,1,TechCorp,Technology,2005,USA,0,1.000000
6,8,AlphaSoft Systems,3800,7000,Spain,6,7,AlphaSoft,Software,2007,France,6,0.783905


In [6]:
df_lm_matched[["CompanyName_x","CompanyName_y","score","Revenue (Millions USD)","Num_Employees"]]

,CompanyName_x,CompanyName_y,score,Revenue (Millions USD),Num_Employees
0,Tech Corporation,TechCorp,0.828613,5000,10000
1,InfoTech Soln,InfoTech Solutions,0.806597,4500,8500
2,GlobalSoft Incorporated,GlobalSoft Inc,0.941341,3000,6000
3,DataTech Corporation,DataTech Co,0.934297,2500,5000
4,SoftSys Limited,SoftSys Ltd,0.811891,4000,7500
5,TechCorp,TechCorp,1.000000,5500,12000
6,AlphaSoft Systems,AlphaSoft,0.783905,3800,7000


## Merge with Blocking - easily in a line

Often it is necessary to merge within partitions of datasets using a "blocking" variable. In this case, we merge "CompanyName" with only companies that have the same "Country" value.


In [7]:
df1 = pd.read_csv("toy_comp_1.csv")
df2 = pd.read_csv("toy_comp_2.csv")

# Test your function here
df_lm_matched = lt.merge_blocking(df2, df1, merge_type='1:m', on="CompanyName", model="all-MiniLM-L6-v2", left_on=None, right_on=None, blocking_vars=["Country"])
df_lm_matched.head(10)


Number of blocks in df1: 6
Number of blocks in df2: 6
2024-10-26 08:24:25 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:27 - Use pytorch device_name: cpu
Merging block UK


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Merging block Canada


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,CompanyID_x,CompanyName_x,Revenue (Millions USD),Num_Employees,Country_x,id_lt_x,CompanyID_y,CompanyName_y,Industry,Founded_Year,Country_y,id_lt_y,score,CompanyID,CompanyName,Country
0,5.0,SoftSys Limited,4000.0,7500.0,UK,0.0,5.0,SoftSys Ltd,Software,2003.0,UK,0.0,0.811891,NaN,NaN,NaN
1,1.0,Tech Corporation,5000.0,10000.0,USA,0.0,1.0,TechCorp,Technology,2005.0,USA,0.0,0.828613,NaN,NaN,NaN
2,6.0,TechCorp,5500.0,12000.0,USA,1.0,1.0,TechCorp,Technology,2005.0,USA,0.0,1.000000,NaN,NaN,NaN
3,3.0,GlobalSoft Incorporated,3000.0,6000.0,India,0.0,3.0,GlobalSoft Inc,Software,2010.0,India,0.0,0.941341,NaN,NaN,NaN
4,4.0,DataTech Corporation,2500.0,5000.0,Germany,0.0,4.0,DataTech Co,Data Analytics,2012.0,Germany,0.0,0.934297,NaN,NaN,NaN
5,2.0,InfoTech Soln,4500.0,8500.0,Canada,0.0,2.0,InfoTech Solutions,Technology,1998.0,Canada,0.0,0.806597,NaN,NaN,NaN
6,NaN,NaN,3800.0,7000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,AlphaSoft Systems,Spain
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Software,2007.0,NaN,NaN,NaN,7.0,AlphaSoft,France


## All functions also support multiple keys!

Matching over multiple keys is also supported, for example, CompanyName and ProductDescription. Multiple columns are serialized by concatenating with a <SEP> token that the package selects to be suitable with the base language model tokenizer.

In [8]:
df1 = pd.read_csv("toy_multi_1.csv")
df2 = pd.read_csv("toy_multi_2.csv")

df1.head(3)

,ProductID,CompanyName,ProductCategory,ProductDescription,Price
0,101,TechCorp,Electronics,High-performance laptop with SSD,1500
1,102,InfoTech Solutions,Electronics,Smartphone with AI capabilities,1000
2,103,GlobalSoft Inc,Software,Enterprise CRM software for businesses,2000


In [9]:
# Test your function here
df_lm_matched = lt.merge(df2, df1, merge_type='1:m', on=["CompanyName", "ProductDescription"], model="all-MiniLM-L6-v2", left_on=None, right_on=None)

df_lm_matched.head(10)

2024-10-26 08:24:27 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:28 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,ProductID_x,CompanyName_x,ProductCategory_x,ProductDescription_x,Price_x,id_lt_x,ProductID_y,CompanyName_y,ProductCategory_y,ProductDescription_y,Price_y,id_lt_y,score
0,201,TechCorp,Electronics,High performance laptop with solid state drive,1400,0,101,TechCorp,Electronics,High-performance laptop with SSD,1500,0,0.847999
1,202,InfoTech Solutions,Electronics,Smartphone with AI support,950,1,102,InfoTech Solutions,Electronics,Smartphone with AI capabilities,1000,1,0.963210
2,203,GlobalSoft,Software,CRM software for enterprises,1800,2,103,GlobalSoft Inc,Software,Enterprise CRM software for businesses,2000,2,0.926065
3,204,DataTech,Data Analysis,Data analytics platform for large datasets,1700,3,104,DataTech Co,Data Analytics,Data analysis platform for big data,1800,3,0.812184
4,205,SoftSys,Software,Document management software for businesses,2000,4,105,SoftSys Ltd,Software,Document management system for enterprises,2200,4,0.918345
5,206,TechCorp,Electronics,Powerful gaming laptop with SSD,1600,5,101,TechCorp,Electronics,High-performance laptop with SSD,1500,0,0.904424
6,207,InfoTech Solutions,Electronics,Smartphone with advanced AI capabilities,1100,6,102,InfoTech Solutions,Electronics,Smartphone with AI capabilities,1000,1,0.967993
7,208,GlobalSoft,Software,Advanced CRM software for businesses,2100,7,103,GlobalSoft Inc,Software,Enterprise CRM software for businesses,2000,2,0.937032
8,209,DataTech,Data Analysis,Big data analytics platform for enterprises,1900,8,104,DataTech Co,Data Analytics,Data analysis platform for big data,1800,3,0.882190
9,210,SoftSys,Software,Enterprise document management system,2300,9,105,SoftSys Ltd,Software,Document management system for enterprises,2200,4,0.915138


## Quick Aggregation or Classification as KNN retrieval
A merge can be used to aggregate or classify (KNN) - say fine product descriptions to coarser categories.

In [10]:
df_coarse = pd.read_csv("coarse.csv")
df_fine = pd.read_csv("fine.csv")

In [11]:

df_lm_aggregate = lt.aggregate_rows(df_fine, df_coarse, model="all-mpnet-base-v2", left_on="Fine Category Name", right_on="Coarse Category Name")

df_lm_aggregate.head()


2024-10-26 08:24:29 - Load pretrained SentenceTransformer: all-mpnet-base-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:31 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,Fine Category ID,Fine Category Name,Coarse Category ID_x,id_lt_x,Coarse Category ID_y,Coarse Category Name,id_lt_y,score
0,101,Laptops,1,0,1,Computers & Electronics,0,0.596126
1,102,Desktop Computers,1,1,1,Computers & Electronics,0,0.551886
2,103,Smartphones,1,2,1,Computers & Electronics,0,0.540363
3,104,Tablets,1,3,1,Computers & Electronics,0,0.352265
4,105,Headphones,1,4,1,Computers & Electronics,0,0.351324


In [12]:
df_lm_aggregate[["Fine Category Name","Coarse Category Name"]].head(10)

,Fine Category Name,Coarse Category Name
0,Laptops,Computers & Electronics
1,Desktop Computers,Computers & Electronics
2,Smartphones,Computers & Electronics
3,Tablets,Computers & Electronics
4,Headphones,Computers & Electronics
5,Speakers,Computers & Electronics
6,Refrigerators,Home Appliances
7,Washing Machines,Home Appliances
8,Vacuum Cleaners,Home Appliances
9,Toasters,Home Appliances


## Bypass translation with Cross-lingual merges
You can merge across languages - without having the need to translate anything! Just use a multilingual model to merge.

In [13]:
df_french = pd.read_csv("translation_1.csv")
df_english = pd.read_csv("translation_2.csv")

df_french.head(3)

,ProductID,Libellé du Produit,Catégorie,Prix
0,101,Ordinateur portable performant,Électronique,1400
1,102,Smartphone avec prise en charge de l'IA,Électronique,950
2,103,Logiciel CRM pour entreprises,Logiciel,1800


In [14]:
# French to English Merge


# Test your function here
df_lm_matched = lt.merge(df_french, df_english, merge_type='1:m', left_on="Libellé du Produit", right_on="Product Label", model="distiluse-base-multilingual-cased-v1")

df_lm_matched.head(10)


2024-10-26 08:24:31 - Load pretrained SentenceTransformer: distiluse-base-multilingual-cased-v1


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:33 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,ProductID_x,Libellé du Produit,Catégorie,Prix,id_lt_x,ProductID_y,Product Label,Category,Price,id_lt_y,score
0,101,Ordinateur portable performant,Électronique,1400,0,101,High-performance laptop,Electronics,1400,0,0.484677
1,102,Smartphone avec prise en charge de l'IA,Électronique,950,1,102,Smartphone with AI support,Electronics,950,1,0.746061
2,103,Logiciel CRM pour entreprises,Logiciel,1800,2,103,CRM software for businesses,Software,1800,2,0.897755
3,104,Plateforme d'analyse de données pour gros ense...,Analyse de données,1700,3,104,Data analytics platform for large datasets,Data Analytics,1700,3,0.934434
4,105,Gestion de documents pour les entreprises,Logiciel,2000,4,105,Document management for businesses,Software,2000,4,0.965277
5,106,Caméra de sécurité à haute résolution,Électronique,1200,5,106,High-resolution security camera,Electronics,1200,5,0.955552
6,107,Imprimante tout-en-un,Électronique,600,6,107,All-in-one printer,Electronics,600,6,0.877312
7,108,Casque audio sans fil,Électronique,300,7,108,Wireless headphones,Electronics,300,7,0.722391
8,109,Système de divertissement à domicile,Électronique,2500,8,109,Home entertainment system,Electronics,2500,8,0.934203
9,110,Robot aspirateur intelligent,Électronique,800,9,110,Intelligent robot vacuum cleaner,Electronics,800,9,0.821990


In [15]:
df_lm_matched[["Libellé du Produit","Product Label","Catégorie","Category"]].head(10)

,Libellé du Produit,Product Label,Catégorie,Category
0,Ordinateur portable performant,High-performance laptop,Électronique,Electronics
1,Smartphone avec prise en charge de l'IA,Smartphone with AI support,Électronique,Electronics
2,Logiciel CRM pour entreprises,CRM software for businesses,Logiciel,Software
3,Plateforme d'analyse de données pour gros ense...,Data analytics platform for large datasets,Analyse de données,Data Analytics
4,Gestion de documents pour les entreprises,Document management for businesses,Logiciel,Software
5,Caméra de sécurité à haute résolution,High-resolution security camera,Électronique,Electronics
6,Imprimante tout-en-un,All-in-one printer,Électronique,Electronics
7,Casque audio sans fil,Wireless headphones,Électronique,Electronics
8,Système de divertissement à domicile,Home entertainment system,Électronique,Electronics
9,Robot aspirateur intelligent,Intelligent robot vacuum cleaner,Électronique,Electronics


## Perform clustering at the row level in one line!

Several algorithms are supported and more to be added soon.


In [16]:
df=pd.read_csv("toy_comp_2.csv")
df_cluster=lt.cluster_rows(df,on="CompanyName",model="sentence-transformers/all-MiniLM-L6-v2",cluster_type= "agglomerative",
    cluster_params= {'threshold': 0.7})
df_cluster.head(10)

2024-10-26 08:24:33 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:35 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,CompanyID,CompanyName,Revenue (Millions USD),Num_Employees,Country,cluster
0,1,Tech Corporation,5000,10000,USA,0
1,2,InfoTech Soln,4500,8500,Canada,5
2,3,GlobalSoft Incorporated,3000,6000,India,3
3,4,DataTech Corporation,2500,5000,Germany,4
4,5,SoftSys Limited,4000,7500,UK,2
5,6,TechCorp,5500,12000,USA,0
6,8,AlphaSoft Systems,3800,7000,Spain,1


## Deduplicate your rows in one line!
Deduplicate using a language model with one line of code! Several clustering algorithms like SLINK, HDBSCAN and DBSCAN are supported

In [17]:
df=pd.read_csv("toy_comp_2.csv")
df_dedup=lt.dedup_rows(df,on="CompanyName",model="sentence-transformers/all-MiniLM-L6-v2",cluster_type= "agglomerative",
    cluster_params= {'threshold': 0.7})
df_dedup.head(10)

2024-10-26 08:24:35 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:36 - Use pytorch device_name: cpu
Deduplicating dataframe with originally 7 rows
Checking for and dropping exact duplicates
Number of rows after dropping exact duplicates: 7


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of rows after deduplication: 6


,CompanyID,CompanyName,Revenue (Millions USD),Num_Employees,Country
0,1,Tech Corporation,5000,10000,USA
1,2,InfoTech Soln,4500,8500,Canada
2,3,GlobalSoft Incorporated,3000,6000,India
3,4,DataTech Corporation,2500,5000,Germany
4,5,SoftSys Limited,4000,7500,UK
6,8,AlphaSoft Systems,3800,7000,Spain


In [18]:
df.head(10)

,CompanyID,CompanyName,Revenue (Millions USD),Num_Employees,Country
0,1,Tech Corporation,5000,10000,USA
1,2,InfoTech Soln,4500,8500,Canada
2,3,GlobalSoft Incorporated,3000,6000,India
3,4,DataTech Corporation,2500,5000,Germany
4,5,SoftSys Limited,4000,7500,UK
5,6,TechCorp,5500,12000,USA
6,8,AlphaSoft Systems,3800,7000,Spain


## We also support getting semantic similarity scores between values of specified columns!
All you need is a dataframe containing the columns of interest as keys, and you are done! This will be a very powerful drop in replacement for edit distance calculations

In [19]:
df=pd.read_csv("toy_pairs.csv")
df.head(3)

,company_name_1,company_name_2,label
0,Acme Corporation,ACME Corporation,1
1,TechGiant Inc.,TechGiant Ltd.,0
2,Widgets R Us,Widgets Incorporated,0


In [20]:

df_eval=lt.evaluate_pairs(df, model="sentence-transformers/all-MiniLM-L6-v2",left_on="company_name_1",right_on="company_name_2",openai_key=None)
df_eval.head()

2024-10-26 08:24:36 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:37 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,company_name_1,company_name_2,label,score
0,Acme Corporation,ACME Corporation,1,1.000000
1,TechGiant Inc.,TechGiant Ltd.,0,0.906331
2,Widgets R Us,Widgets Incorporated,0,0.780843
3,SuperTech Solutions,SuperTech Corp,1,0.758345
4,Globex Industries,Globex Corp,1,0.918245


## Get all pair-wise distances between keys from two datasets

In [21]:
df=pd.read_csv("toy_pairs.csv")
df_eval=lt.all_pair_combos_evaluate(df, model="sentence-transformers/all-MiniLM-L6-v2",left_on="company_name_1",right_on="company_name_2",openai_key=None)
df_eval.head(100)

2024-10-26 08:24:37 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:39 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,left_on,right_on,score
0,Acme Corporation,ACME Corporation,1.000000
1,Acme Corporation,TechGiant Ltd.,0.451013
2,Acme Corporation,Widgets Incorporated,0.184706
3,Acme Corporation,SuperTech Corp,0.514488
4,Acme Corporation,Globex Corp,0.458063
...,...,...,...
95,BestWare Inc.,Innovations Unlimited,0.353746
96,BestWare Inc.,AlphaTech Systems,0.292267
97,BestWare Inc.,ABC Solutions,0.060783
98,BestWare Inc.,TechMaster Corp.,0.517866


## Get K nearest matches for each row in the left df

In [22]:
df1 = pd.read_csv("toy_comp_1.csv")
df2 = pd.read_csv("toy_comp_2.csv")

df_lm_matched_2nn = lt.merge_knn(df2, df1, on="CompanyName", model="sentence-transformers/all-MiniLM-L6-v2", left_on=None, right_on=None,
                                k=2)

df_lm_matched_2nn.head()

2024-10-26 08:24:39 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:24:40 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Adding embeddings to index
Searching index
LM matched on key columns - left: CompanyName_x, right: CompanyName_y


,CompanyID_x,CompanyName_x,Revenue (Millions USD),Num_Employees,Country_x,id_lt_x,CompanyID_y,CompanyName_y,Industry,Founded_Year,Country_y,id_lt_y,score
0,1,Tech Corporation,5000,10000,USA,0,6,TechCorp,Technology,2005,USA,5,0.828613
1,1,Tech Corporation,5000,10000,USA,0,1,TechCorp,Technology,2005,USA,0,0.828613
2,2,InfoTech Soln,4500,8500,Canada,1,2,InfoTech Solutions,Technology,1998,Canada,1,0.806597
3,2,InfoTech Soln,4500,8500,Canada,1,4,DataTech Co,Data Analytics,2012,Germany,3,0.641552
4,3,GlobalSoft Incorporated,3000,6000,India,2,3,GlobalSoft Inc,Software,2010,India,2,0.941341


In [23]:
df_lm_matched = lt.merge(df1, df2, merge_type='1:m', on="CompanyName", model="dell-research-harvard/lt-wikidata-comp-en", left_on=None, right_on=None)

df_lm_matched.head(10)

2024-10-26 08:25:45 - Load pretrained SentenceTransformer: dell-research-harvard/lt-wikidata-comp-en


c:\Users\chadi\anaconda3\envs\islp\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-10-26 08:25:47 - Use pytorch device_name: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,CompanyID_x,CompanyName_x,Industry,Founded_Year,Country_x,id_lt_x,CompanyID_y,CompanyName_y,Revenue (Millions USD),Num_Employees,Country_y,id_lt_y,score
0,1,TechCorp,Technology,2005,USA,0,6,TechCorp,5500,12000,USA,5,1.000000
1,2,InfoTech Solutions,Technology,1998,Canada,1,2,InfoTech Soln,4500,8500,Canada,1,0.707271
2,3,GlobalSoft Inc,Software,2010,India,2,3,GlobalSoft Incorporated,3000,6000,India,2,0.991263
3,4,DataTech Co,Data Analytics,2012,Germany,3,4,DataTech Corporation,2500,5000,Germany,3,0.991782
4,5,SoftSys Ltd,Software,2003,UK,4,5,SoftSys Limited,4000,7500,UK,4,0.997345
5,6,TechCorp,Technology,2005,USA,5,6,TechCorp,5500,12000,USA,5,1.000000
6,7,AlphaSoft,Software,2007,France,6,8,AlphaSoft Systems,3800,7000,Spain,6,0.971633


In [24]:
#save
df_lm_matched.to_csv("toy_comp_matched.csv",index=False)